In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — this cell is intentionally immediately after Drive mount.
from pathlib import Path
CACHE_ROOT = '/content/drive/MyDrive/OpenPlaque/TotalSegmentator_Cardiovascular_Cache_v1'
SOURCE_CACHE = '/content/drive/MyDrive/OpenPlaque/Cache/Secondary_3D_Vesselness_Topology_v1'
PRIOR_BATCH_ROOT = '/content/drive/MyDrive/OpenPlaque/GPU_Batch_Pipeline_v1'
LICENSE_FILE = Path('/content/drive/MyDrive/OpenPlaque/private/totalseg_license.txt')
REUSE = True
IMPORT_PRIOR_BATCH = True
SAVE_PROBABILITIES = False

# Cardiovascular tasks requested from the current UI plus useful context/legacy tasks.
TASKS = [
    'total',
    'total_highres',
    'heartchambers_highres',
    'coronary_arteries',
    'coronary_arteries_LEGACY',
    'aortic_sinuses',
    'aorta_annulus',
    'aortic_dissection',
    'pulmonary_artery_landmarks',
]

# Optional probability caches can be large. Leave empty unless needed later.
PROBABILITY_TASKS = [] if not SAVE_PROBABILITIES else ['coronary_arteries','aortic_dissection']
TOTALSEG_LICENSE = LICENSE_FILE.read_text().strip() if LICENSE_FILE.exists() else ''
print('Drive license file found:', bool(TOTALSEG_LICENSE))


# OpenPlaque — TotalSegmentator cardiovascular cache

Runs cardiovascular/heart TotalSegmentator tasks serially and permanently caches every mask to Drive. Existing results from `GPU_Batch_Pipeline_v1` are imported instead of recomputed. Each task has its own folder and `_SUCCESS` marker, so Runtime → Run all is safe after a disconnect.

**Important:** task support is taken from the installed TotalSegmentator registry at runtime. If a task shown by a web picker (for example `total_highres`) is not exposed by the installed CLI, it is logged as unsupported rather than silently substituted.


In [ ]:
!pip -q install TotalSegmentator SimpleITK pandas numpy
import sys, shutil, subprocess
from pathlib import Path
repo=Path('/content/OpenPlaque')
if repo.exists(): shutil.rmtree(repo)
!git clone -q --depth 1 --branch totalseg-cardiovascular-cache-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0,'/content/OpenPlaque/src')
!nvidia-smi

if TOTALSEG_LICENSE:
    subprocess.run(['totalseg_set_license','-l',TOTALSEG_LICENSE], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    print('TotalSegmentator license activated for this runtime.')
else:
    print('No Drive license file found; licensed tasks will be skipped.')


In [ ]:
from totalsegmentator.registry import format_tasks_table
print(format_tasks_table())


In [ ]:
from openplaque.totalseg_cardiovascular_cache import run_cache
cfg = dict(
    CACHE_ROOT=CACHE_ROOT,
    SOURCE_CACHE=SOURCE_CACHE,
    PRIOR_BATCH_ROOT=PRIOR_BATCH_ROOT,
    TASKS=TASKS,
    REUSE=REUSE,
    IMPORT_PRIOR_BATCH=IMPORT_PRIOR_BATCH,
    PROBABILITY_TASKS=PROBABILITY_TASKS,
    LICENSE_ACTIVE=bool(TOTALSEG_LICENSE),
)
summary, report_zip = run_cache(cfg)
print(summary)
print('REPORT ZIP:', report_zip)
print('CACHE ROOT:', CACHE_ROOT)
